<a href="https://colab.research.google.com/github/jagdish1013/jagdish1013-CIFAR10-CNN-Mode/blob/main/CIFAR10_CNN_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

#this is offical library for computer vision(cv) and it can have many
#dataset (cifr10,mnist), pretrained cnn, utlies for image transform
import torchvision

from torchvision.datasets import CIFAR10

In [2]:
#Datasets & DataLoaders
from torch.utils.data import DataLoader
import torchvision.transforms as transforms #this are help to perform transformation on image

# image => scale (0,1) => normalize => (-1,1)
#in compose function we are applyed transfromation prewrite so we are use after in image
# ToTenser are automatically convert image into torch tensor + scall all the images
# after normalize we define which standerd divation and min are required and the we need range is (-1,1) so put (0.5)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = CIFAR10(root="./data", train=True, download=True, transform=transform)
testset = CIFAR10(root="./data", train=False, download=True, transform=transform)


100%|██████████| 170M/170M [26:44<00:00, 106kB/s] 


In [3]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [4]:
trainloader = DataLoader(trainset, batch_size = 64, shuffle=True)
testloader = DataLoader(testset, batch_size = 64)

In [5]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()# call the constructor(__init__)

        #convalutional layer (=3)
        #1st
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding = 1),#conv2D(input size, output size, kernal size, padding size)
            nn.ReLU(),
            nn.MaxPool2d(2, 2),# ( kernal size, stride )


        #2nd
            nn.Conv2d(32, 64, kernel_size=3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),


        #3rd

            nn.Conv2d(64, 128, kernel_size=3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )

        #fully connected layer
        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256), # o/p image are alredy flatten and the value is 4*4*128 and the 256 in needed o/p neurons
            nn.ReLU(),

            nn.Linear(256,10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # for flattening
        x = self.fc_layers(x)

        return x;


In [6]:
model = CNN()

In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

**Training the CNN**

In [8]:
epochs = 10

for epoch in range (epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()

        output = model.forward(images)#forward propogation
        loss = criterion(output, labels)# loss fux
        loss.backward() #backward perpogation
        optimizer.step() #update parameters

        epoch_training_loss += loss.item() # .item because it convert loss tensor data to float data

    print(f"epoch={epoch+1}/{epochs} & loss = {epoch_training_loss/len(trainloader)}")


epoch=1/10 & loss = 1.3780569325932457
epoch=2/10 & loss = 0.9418925035487661
epoch=3/10 & loss = 0.7583218843811919
epoch=4/10 & loss = 0.6262797762823227
epoch=5/10 & loss = 0.5280546499297137
epoch=6/10 & loss = 0.4281899102432344
epoch=7/10 & loss = 0.34854127646750194
epoch=8/10 & loss = 0.272596718471907
epoch=9/10 & loss = 0.2182544384132642
epoch=10/10 & loss = 0.16730744299976646


**Evaluate our CNN**

In [12]:


correct_labels = 0
total_labels = 0
model.eval()

#no_grad because we not do backward perpogation for update the value
with torch.no_grad():
  for images, labels in testloader:
    output = model.forward(images)
    _, predicated = torch.max(output, 1)

    correct_labels += (predicated == labels).sum().item()
    total_labels += labels.size(0)

print(f"accuracy = {correct_labels / total_labels *100}")

accuracy = 74.76
